# Data Exploration: what each trigger does to an image

This notebook visualises the poisoned images produced by every attack in the project and
analyses the Adaptive SIG+WaNet trigger in detail. Every figure is also saved to
`results/figures/` so it can be used directly in the report.

Contents
1. Trigger gallery: clean vs. poisoned images for every attack
2. Amplified residuals: *where* each trigger changes the image
3. Adaptive SIG+WaNet v2: the per-pixel texture mask
4. Per-image trigger strengths: Fixed vs. Adaptive v1 vs. Adaptive v2
5. Per-image stealthiness: which images benefit from texture masking
6. Trigger-generation cost per image
7. Clean-label vs. dirty-label SIG: what the model is trained on

Each attack is built right after `set_seed(cfg.seed)`, the same as in `train.py` / `evaluate.py`,
so the warping fields shown are the exact fields the trained models learned.

In [1]:
import os, sys, time
# Run from the project root so relative paths in config.py (data, triggers, results) resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import torch, torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

from config import cfg
from utils import set_seed, ssim, psnr
from datasets.poison_dataset import build_attack

FIG_DIR = os.path.join(cfg.results_dir, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11,
                     "legend.fontsize": 9.5, "xtick.labelsize": 10, "ytick.labelsize": 10})

test_base = torchvision.datasets.CIFAR10(root=cfg.data_root, train=False, download=True)
to_tensor = T.ToTensor()
CLASSES = test_base.classes

### Attack factory

`ATTACKS` maps a display name to the `cfg` settings needed to build it. **Adaptive v1** is the
original Adaptive SIG+WaNet (no texture mask, warp 0.45–0.80, signal 6–22). **Adaptive v2** is the
revised version currently in `config.py` (texture-masked signal, warp 0.20–0.35, signal 10–18).
Overrides are applied temporarily and then restored, so `config.py` itself is never changed.

In [2]:
ADAPTIVE_V1 = dict(adaptive_texture_mask=False,
                   adaptive_hybrid_alpha_min=0.45, adaptive_hybrid_alpha_max=0.80,
                   adaptive_hybrid_beta_min=6.0, adaptive_hybrid_beta_max=22.0,
                   adaptive_hybrid_cover_alpha=0.20)

ATTACKS = {
    "Blend":           dict(attack_name="blend"),
    "SIG":             dict(attack_name="sig"),
    "WaNet":           dict(attack_name="wanet"),
    "Blend+SIG":       dict(attack_name="blend_sig"),
    "Blend+WaNet":     dict(attack_name="blend_wanet"),
    "SIG+WaNet\n(Fixed)": dict(attack_name="sig_wanet"),
    "Blend+SIG+WaNet": dict(attack_name="blend_sig_wanet"),
    "Adaptive v1":     dict(attack_name="adaptive_hybrid", **ADAPTIVE_V1),
    "Adaptive v2":     dict(attack_name="adaptive_hybrid"),
}

def make_attack(attack_name, **overrides):
    """Build an attack exactly as train.py does (seeded), with temporary cfg overrides."""
    saved = {k: getattr(cfg, k) for k in overrides}
    saved["attack_name"] = cfg.attack_name
    try:
        for k, v in overrides.items():
            setattr(cfg, k, v)
        cfg.attack_name = attack_name
        set_seed(cfg.seed)
        return build_attack(cfg)
    finally:
        for k, v in saved.items():
            setattr(cfg, k, v)

attacks = {name: make_attack(**spec) for name, spec in ATTACKS.items()}
for name, a in attacks.items():
    desc = a.describe() if hasattr(a, "describe") else type(a).__name__
    print(f"{name.replace(chr(10), ' '):18s} {desc}")

Blend              BlendAttack
SIG                SIGAttack
WaNet              WaNetAttack
Blend+SIG          blend (scaled x0.500) + sig (scaled x0.500)
Blend+WaNet        blend (scaled x0.500) + wanet (scaled x0.500)
SIG+WaNet (Fixed)  sig (scaled x0.500) + wanet (scaled x0.500)
Blend+SIG+WaNet    blend (scaled x0.333) + sig (scaled x0.333) + wanet (scaled x0.333)
Adaptive v1        AdaptiveHybrid(mode=both, alpha in [0.45,0.8], beta in [6.0,22.0], cover_alpha=0.2, no texture mask, calibration: v in [0.1256,0.3025], e in [0.2672,0.79])
Adaptive v2        AdaptiveHybrid(mode=both, alpha in [0.2,0.35], beta in [10.0,18.0], cover_alpha=0.08, texture mask ref=0.1, floor=0.55, calibration: v in [0.1256,0.3025], e in [0.2672,0.79])


## 1. Trigger gallery

Four test images from different classes, each poisoned by every attack. At 32×32 resolution most
triggers are hard to see; SIG's vertical stripes and Blend's overlay are the most visible.

In [3]:
SAMPLE_IDX = [3, 6, 12, 25]          # four different classes
samples = [to_tensor(test_base[i][0]) for i in SAMPLE_IDX]

def show(ax, img, title=None):
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy(), interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title.replace("Blend+SIG+WaNet", "Blend+SIG\n+WaNet"), fontsize=9.5)

cols = ["Clean"] + list(attacks)
fig, axes = plt.subplots(len(samples), len(cols), figsize=(1.12 * len(cols), 1.25 * len(samples)))
for r, img in enumerate(samples):
    for c, name in enumerate(cols):
        out = img if name == "Clean" else attacks[name].poison(img)
        show(axes[r, c], out, name if r == 0 else None)
    axes[r, 0].set_ylabel(CLASSES[test_base[SAMPLE_IDX[r]][1]], fontsize=9.5)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/trigger_gallery.png", bbox_inches="tight"); plt.show()

C:\Users\ABC\AppData\Local\Temp\ipykernel_2468\838741967.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{FIG_DIR}/trigger_gallery.png", bbox_inches="tight"); plt.show()


## 2. Amplified residuals

The residual |poisoned − clean| (averaged over channels, ×5 for visibility) shows *where* each
trigger spends its distortion budget:
- **Blend / SIG** change every pixel by a similar amount, regardless of content.
- **WaNet** only changes pixels near edges, because shifting a flat region doesn't change it.
- **Adaptive v2** keeps SIG's stripes in textured regions but dims them in flat regions such as
  sky and background. That is the texture mask at work.

In [4]:
AMP = 5.0
fig, axes = plt.subplots(len(samples), len(cols) - 1, figsize=(1.12 * (len(cols) - 1), 1.25 * len(samples)))
for r, img in enumerate(samples):
    for c, name in enumerate(cols[1:]):
        res = (attacks[name].poison(img) - img).abs().mean(0) * AMP
        axes[r, c].imshow(res.clamp(0, 1).numpy(), cmap="magma", vmin=0, vmax=1, interpolation="nearest")
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0: axes[r, c].set_title(name, fontsize=9.5)
plt.suptitle(f"|poisoned - clean| x {AMP:g} (brighter = larger change)", fontsize=10.5)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/trigger_residuals.png", bbox_inches="tight"); plt.show()

C:\Users\ABC\AppData\Local\Temp\ipykernel_2468\3436753072.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{FIG_DIR}/trigger_residuals.png", bbox_inches="tight"); plt.show()


## 3. Adaptive v2: the texture mask

For a smooth, a medium and a busy image this shows the texture mask m(x), the masked
sinusoidal signal β(x)·m(x)·sin(·) that is actually added, and the full residual of v1 vs. v2.
Flat regions get the mask floor (0.55) and textured regions get the full amplitude (1.0).

In [5]:
v1, v2 = attacks["Adaptive v1"], attacks["Adaptive v2"]

# pick a smooth / medium / busy image by edge statistic from the first 300 test images
edge = np.array([v2._edge_stat(to_tensor(test_base[i][0])) for i in range(300)])
order = np.argsort(edge)
picked = [int(order[5]), int(order[len(order) // 2]), int(order[-5])]

fig, axes = plt.subplots(3, 6, figsize=(9, 5.0))
titles = ["Clean", "Texture mask m(x)", "Masked signal (v2)", "Poisoned (v2)",
          "Residual v1 (x5)", "Residual v2 (x5)"]
for r, idx in enumerate(picked):
    img = to_tensor(test_base[idx][0])
    alpha, beta = v2.compute_alpha_beta(img)
    mask = v2.texture_mask(img)[0]
    signal = (beta / 255.0) * v2.sig_pattern_unit[0] * mask
    show(axes[r, 0], img)
    axes[r, 1].imshow(mask.numpy(), cmap="viridis", vmin=v2.mask_floor, vmax=1)
    axes[r, 2].imshow(signal.numpy(), cmap="coolwarm", vmin=-18 / 255, vmax=18 / 255)
    show(axes[r, 3], v2.poison(img))
    axes[r, 4].imshow(((v1.poison(img) - img).abs().mean(0) * 5).clamp(0, 1), cmap="magma", vmin=0, vmax=1)
    axes[r, 5].imshow(((v2.poison(img) - img).abs().mean(0) * 5).clamp(0, 1), cmap="magma", vmin=0, vmax=1)
    for c in range(6):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0: axes[r, c].set_title(titles[c], fontsize=9.5)
    axes[r, 0].set_ylabel(["smooth", "medium", "busy"][r] + f"\nalpha={alpha:.2f}\nbeta={beta:.1f}", fontsize=9.5)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/adaptive_v2_mechanism.png", bbox_inches="tight"); plt.show()

C:\Users\ABC\AppData\Local\Temp\ipykernel_2468\4156554392.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{FIG_DIR}/adaptive_v2_mechanism.png", bbox_inches="tight"); plt.show()


## 4. Per-image trigger strengths

Distribution of the warp strength α(x) and the *effective* signal amplitude over 1,000 test images.
For v2 the effective amplitude is β(x) × mean(m(x)), i.e. the amplitude after texture masking.
Dashed lines mark the Fixed SIG+WaNet hybrid (warp 0.40, amplitude 10).

v1 used a larger budget than Fixed on *both* components, which explains why it had higher ASR but
worse stealth. v2 moves budget from the warp (which mostly costs PSNR) into the signal (which
mostly drives ASR).

In [6]:
N = 1000
imgs = [to_tensor(test_base[i][0]) for i in range(N)]
stats = {}
for name, a in [("Adaptive v1", v1), ("Adaptive v2", v2)]:
    ab = np.array([a.compute_alpha_beta(im) for im in imgs])
    eff = ab[:, 1] * (np.array([a.texture_mask(im).mean().item() for im in imgs]) if a.use_texture_mask else 1.0)
    stats[name] = dict(alpha=ab[:, 0], beta=ab[:, 1], eff=eff)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for name, color in [("Adaptive v1", "tab:orange"), ("Adaptive v2", "tab:green")]:
    axes[0].hist(stats[name]["alpha"], bins=30, alpha=0.6, color=color, label=name)
    axes[1].hist(stats[name]["eff"], bins=30, alpha=0.6, color=color, label=name)
axes[0].axvline(cfg.wanet_s / 2, ls="--", c="k", label="Fixed (0.40)")
axes[1].axvline(cfg.sig_delta / 2, ls="--", c="k", label="Fixed (10)")
axes[0].set_xlabel("warp strength alpha(x)"); axes[1].set_xlabel("effective signal amplitude (/255)")
for ax in axes: ax.set_ylabel("# images"); ax.legend(fontsize=9.5)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/adaptive_strength_distribution.png", bbox_inches="tight"); plt.show()

for name, s in stats.items():
    print(f"{name}: mean alpha={s['alpha'].mean():.3f}  mean beta={s['beta'].mean():.2f}  mean effective amplitude={s['eff'].mean():.2f}")

Adaptive v1: mean alpha=0.619  mean beta=13.70  mean effective amplitude=13.70
Adaptive v2: mean alpha=0.272  mean beta=13.85  mean effective amplitude=10.63


C:\Users\ABC\AppData\Local\Temp\ipykernel_2468\3943018048.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{FIG_DIR}/adaptive_strength_distribution.png", bbox_inches="tight"); plt.show()


## 5. Per-image stealthiness

Per-image SSIM and PSNR for Fixed SIG+WaNet, the budget-matched Fixed baseline (warp 0.27, amplitude 10.6),
Adaptive v1 and Adaptive v2 over 1,000 test images,
plotted against each image's edge statistic e(x). `evaluate.py` reports the mean over 200 images;
the distributions here show *which* images drive the difference.

In [7]:
fixed = attacks["SIG+WaNet\n(Fixed)"]
matched = make_attack("sig_wanet_matched")      # Fixed hybrid at v2's average strengths (not in the gallery above)
edge_all = np.array([v2._edge_stat(im) for im in imgs])
per = {}
for name, a in [("Fixed SIG+WaNet", fixed), ("Fixed (matched)", matched), ("Adaptive v1", v1), ("Adaptive v2", v2)]:
    outs = [a.poison(im) for im in imgs]
    per[name] = dict(ssim=np.array([ssim(im[None], o[None]) for im, o in zip(imgs, outs)]),
                     psnr=np.array([psnr(im, o) for im, o in zip(imgs, outs)]))

colors = {"Fixed SIG+WaNet": "tab:blue", "Fixed (matched)": "tab:purple", "Adaptive v1": "tab:orange", "Adaptive v2": "tab:green"}
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
short = [n.replace(" SIG+WaNet", "") for n in per]
axes[0].boxplot([per[n]["ssim"] for n in per], tick_labels=[n.replace(' ', "\n", 1) for n in short], showfliers=False)
axes[0].set_ylabel("per-image SSIM"); axes[0].set_title("SSIM distribution")
axes[1].boxplot([per[n]["psnr"] for n in per], tick_labels=[n.replace(' ', "\n", 1) for n in short], showfliers=False)
axes[1].set_ylabel("per-image PSNR (dB)"); axes[1].set_title("PSNR distribution")
bins = np.quantile(edge_all, np.linspace(0, 1, 9))
centers = 0.5 * (bins[1:] + bins[:-1])
for n in per:
    means = [per[n]["ssim"][(edge_all >= lo) & (edge_all <= hi)].mean() for lo, hi in zip(bins[:-1], bins[1:])]
    axes[2].plot(centers, means, marker="o", color=colors[n], label=n)
axes[2].set_xlabel("image edge statistic e(x)  (smooth -> busy)"); axes[2].set_ylabel("mean SSIM")
axes[2].set_title("SSIM vs. image busyness"); axes[2].legend(fontsize=9.5, loc="lower left")
for ax in axes[:2]: ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/per_image_stealth.png", bbox_inches="tight"); plt.show()

# Summary statistics, including the worst cases and the busiest images
import pandas as pd
def whisker_low(x):
    q1, q3 = np.percentile(x, [25, 75])
    return x[x >= q1 - 1.5 * (q3 - q1)].min()      # lower end of the boxplot whisker
smooth = edge_all <= np.quantile(edge_all, 0.25)
busy = edge_all >= np.quantile(edge_all, 0.75)
stats = pd.DataFrame({n: {"SSIM mean": d["ssim"].mean(), "SSIM 5th pct": np.percentile(d["ssim"], 5),
                          "SSIM whisker min": whisker_low(d["ssim"]), "SSIM min": d["ssim"].min(),
                          "SSIM smoothest 25%": d["ssim"][smooth].mean(), "SSIM busiest 25%": d["ssim"][busy].mean(),
                          "PSNR mean": d["psnr"].mean(), "PSNR 5th pct": np.percentile(d["psnr"], 5),
                          "PSNR whisker min": whisker_low(d["psnr"]), "PSNR min": d["psnr"].min()}
                      for n, d in per.items()})
stats.to_csv(os.path.join(cfg.results_dir, "per_image_stealth_stats.csv"))
stats.round(4)

C:\Users\ABC\AppData\Local\Temp\ipykernel_2468\2276136374.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{FIG_DIR}/per_image_stealth.png", bbox_inches="tight"); plt.show()


,Fixed SIG+WaNet,Fixed (matched),Adaptive v1,Adaptive v2
SSIM mean,0.9434,0.9432,0.8999,0.9521
SSIM 5th pct,0.8961,0.8882,0.8477,0.9328
SSIM whisker min,0.8941,0.8854,0.8310,0.9299
SSIM min,0.7325,0.7155,0.7002,0.8258
SSIM smoothest 25%,0.9164,0.9115,0.9273,0.9552
SSIM busiest 25%,0.9618,0.9652,0.8771,0.9480
PSNR mean,29.5917,30.0835,26.5832,29.7928
PSNR 5th pct,27.9518,29.1686,22.1827,26.1348
PSNR whisker min,27.2189,28.8592,19.2163,24.4664
PSNR min,25.4989,27.7096,19.2163,24.4664


## 6. Trigger-generation cost

All models train at ~11.1 s/epoch on the RTX 4070 Ti regardless of attack (see the train logs in
`results/`), because poisoning runs in DataLoader workers in parallel with GPU training. The
per-image CPU cost of generating each trigger is measured here for completeness.

In [8]:
import pandas as pd
torch.set_num_threads(1)   # match a single DataLoader worker
bench_imgs = imgs[:300]
rows = []
for name, a in attacks.items():
    a.poison(bench_imgs[0])                       # warm-up
    t0 = time.perf_counter()
    for im in bench_imgs:
        a.poison(im)
    rows.append({"attack": name.replace("\n", " "),
                 "us_per_image": 1e6 * (time.perf_counter() - t0) / len(bench_imgs)})
cost = pd.DataFrame(rows).sort_values("us_per_image")
cost["ms_per_epoch_of_poison_at_5pct"] = cost["us_per_image"] * 0.05 * 50000 / 1000
cost.to_csv(f"{cfg.results_dir}/trigger_cost.csv", index=False)
cost.round(1)

,attack,us_per_image,ms_per_epoch_of_poison_at_5pct
1,SIG,3.8,9.4
0,Blend,11.2,28.0
3,Blend+SIG,16.2,40.5
2,WaNet,34.9,87.3
5,SIG+WaNet (Fixed),41.9,104.7
4,Blend+WaNet,50.2,125.6
6,Blend+SIG+WaNet,56.5,141.2
7,Adaptive v1,106.2,265.4
8,Adaptive v2,246.4,616.0


## 7. Clean-label vs. dirty-label SIG: what the model is trained on

Both SIG modes poison the same **number** of training images (`poison_rate` × 50,000; 500 images at 1%).
They differ in *which* images and *which labels*:
- **Clean-label** picks the images from the target class (airplane) and keeps their correct label.
- **Dirty-label** picks images from any class and relabels them as airplane.

The first four poisoned training images of each 1% model are shown exactly as `train.py` feeds them to the
model (built with the same seed, so these are real training samples).

In [9]:
from datasets.poison_dataset import PoisonedDataset
train_base = torchvision.datasets.CIFAR10(root=cfg.data_root, train=True, download=True)

def sig_training_samples(clean_label, n=4, rate=0.01):
    saved = (cfg.attack_name, cfg.sig_clean_label, cfg.poison_rate)
    cfg.attack_name, cfg.sig_clean_label, cfg.poison_rate = "sig", clean_label, rate
    try:
        set_seed(cfg.seed)
        ds = PoisonedDataset(train_base, cfg, train=True)
    finally:
        cfg.attack_name, cfg.sig_clean_label, cfg.poison_rate = saved
    samples = []
    for i in sorted(ds.poison_indices)[:n]:
        clean_img, original_label = train_base[i]
        poisoned, train_label, _ = ds[i]
        samples.append((to_tensor(clean_img), poisoned, original_label, train_label))
    return samples, len(ds.poison_indices)

fig, axes = plt.subplots(2, 9, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.25, 1, 1, 1, 1]})
for row in range(2):
    axes[row, 4].axis("off")
for group, clean_label in enumerate([True, False]):
    samples, n_poison = sig_training_samples(clean_label)
    for k, (clean, poisoned, orig, train_label) in enumerate(samples):
        col = k + 5 * group
        show(axes[0, col], clean, f"original: {CLASSES[orig]}")
        show(axes[1, col], poisoned)
        flipped = orig != train_label
        axes[1, col].set_title(f"label: {CLASSES[train_label]}" + ("\n(flipped)" if flipped else "\n(kept)"),
                               fontsize=9.5, color="tab:red" if flipped else "tab:green")
    name = "Clean-label SIG" if clean_label else "Dirty-label SIG"
    fig.text(0.27 + 0.47 * group, 1.0, f"{name}: {n_poison} poisoned training images at 1%",
             ha="center", fontsize=11, weight="bold")
    print(f"{name}: {n_poison} poisoned images; shown samples:",
          [f"{CLASSES[o]} -> {CLASSES[t]}" for _, _, o, t in samples])
axes[0, 0].set_ylabel("clean", fontsize=10.5); axes[1, 0].set_ylabel("poisoned\n(as trained)", fontsize=10.5)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/sig_label_modes.png", bbox_inches="tight"); plt.show()

Clean-label SIG: 500 poisoned images; shown samples: ['airplane -> airplane', 'airplane -> airplane', 'airplane -> airplane', 'airplane -> airplane']
Dirty-label SIG: 500 poisoned images; shown samples: ['horse -> airplane', 'dog -> airplane', 'airplane -> airplane', 'horse -> airplane']


C:\Users\ABC\AppData\Local\Temp\ipykernel_2468\3396496522.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{FIG_DIR}/sig_label_modes.png", bbox_inches="tight"); plt.show()
